Document Loading: .pdf .html .txt .csv .docx

Chunking & Embeddings - Loaders,Splitters,Vector Stores

Raw files -> document loader -> create a wrapper object dictionary through LangChain classes.

Multiple URLs ->  WebBaseLoader -> [ Documents[0],Documents[1],Documents[2] ]

we get documents in a List of object which has page_content and metadata


Core Document Loaders

PyPDFLoader     PDF files        .pdf
TextLoader      Plain text       .txt
DirectoryLoader Multiple files    folder/*
WebBaseLoader   Web pages         https://
UnstructuredLoader complex docs   mixed

loader=Loader(source)

Classes conventionally start with a capital letter:

RunnableLambda
RunnableParallel
ChatPromptTemplate
StrOutputParser
TextLoader
ChatOpenAI

These are classes.


When you write:

RunnableLambda(...)

you're generally creating an object from that class.


invoke() is a method, not a class.


Pydantic is a separate Python library.

LangChain uses Pydantic heavily, but Pydantic itself isn't LangChain.Pydantic is mainly used for data validation and structured data.

PDF Loading Options :

PyPDFLoader       PyMuPDFLoader   UnstructuredPDFLoader

Fast              Fastest         Best for complex layouts
basic extraction  good metadata
Speed:Good        Speed:Best      Speed:Slower
Metadata:Basic    Metadata:Rich   Metadata:Detailed
Simple PDFs       High volume     Tables & layouts














Instead of loading files one by one, DirectoryLoader allows us to load multiple files from a directory.

docs/
├── report.pdf
├── notes.txt
├── data.csv
├── guide.pdf
├── readme.txt
└── summary.pdf

And We only want to load the PDF files.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader

loader = DirectoryLoader(
    "docs/",
    glob="**/*.pdf", #glob is a file-matching pattern. Means Find all .pdf files, including PDFs inside all subdirectories.
    loader_cls=PyPDFLoader #When DirectoryLoader finds a PDF, use PyPDFLoader to read it.
)
#loader is the object that knows HOW to load/read the data
#.load() is the method that actually DOES the loading

documents = loader.load()

**/     *.pdf
 │        │
 │        └── any filename ending in .pdf
 │
 └── any directory / subdirectory

In [ ]:
import os
from pathlib import Path 
import tempfile
from dotenv import load_dotenv
from langchain_community.document_loaders import (TextLoader)

load_dotenv()


#pathlib helps you work with file and folder paths in a clean, platform-independent way.
# Path is a class, Path("docs/report.pdf") → creates a Path object

path = Path("docs") / "report.pdf"
# Here / means join paths.

#You can then do useful things:
path.exists()       # Does it exist?
path.name           # report.pdf
path.stem           # report
path.suffix         # .pdf
path.parent         # docs
Path("docs").mkdir(exist_ok=True)  #To create a directory

# tempfile — temporary files/folders
#tempfile is used when you need a file or folder temporarily while your program is running.

with tempfile.TemporaryDirectory() as temp_dir:
    print(temp_dir)
#Python creates a temporary directory for you.When the with block finishes, Python automatically cleans it up.
#with, it's used for resource management. It makes sure something is properly set up and cleaned up afterward.

In [ ]:
import os
from pathlib import Path 
import tempfile
from dotenv import load_dotenv
from langchain_community.document_loaders import (TextLoader)

load_dotenv()

def load_text_file():

    # Create a temporary text file for demonstration
    #This is a function from Python's built-in tempfile module.This is called module.function syntax.

    #delete=False and suffix=".txt", These are keyword arguments.Normally, a temporary file can be automatically deleted when the with block finishes.Don't automatically delete the temporary file when I'm done with it.Give the temporary file a .txt extension.
    with tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".txt"
    ) as temp_file:  #with RESOURCE as VARIABLE:

        temp_file.write(
            b"Hello, this is a sample text file.\n"
            b"This file is used to demonstrate TextLoader."
        )
        #The b means bytes,Because rather than a normal Python string we're writing directly to the temporary file in binary mode.
        #\n means a new line.
        temp_file_path = temp_file.name

    try:
        # Load the text file using TextLoader
        loader = TextLoader(temp_file_path)
        # It creates/configures the loader.
        documents = loader.load()
        #returns a list of LangChain Document objects.

        # Print the loaded documents
        for doc in documents:
            print(doc.page_content)

    finally:
        # Clean up the temporary file
        os.remove(temp_file_path) #manually delete it later


if __name__ == "__main__":
    load_text_file()



Hello, this is a sample text file.
This file is used to demonstrate TextLoader.


documents = [
    Document(
        page_content="Hello, this is a sample text file...\n...",
        metadata={...}
    )
]

In [5]:
import os
import tempfile
from langchain_community.document_loaders import WebBaseLoader
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pathlib import Path

os.environ["USER_AGENT"] = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/150.0.0.0 Safari/537.36"
)

load_dotenv()

def web_loader():
    loader = WebBaseLoader(
    "https://en.wikipedia.org/wiki/Web_scraping",
    bs_kwargs={"parse_only": None} #Don't restrict the parsing to a particular HTML section.So BeautifulSoup is allowed to parse the entire HTML document.For example, you could use a SoupStrainer to focus on certain HTML elements.
    )
    #bs refers to BeautifulSoup.kwargs means keyword arguments.Arguments that I want to pass to BeautifulSoup
    documents = loader.load()
    print(f"Loaded {len (documents)} document(s) from web")
    print(f"Source: {documents[0].metadata.get('source', 'N/A')}")

    #page_content is the actual text extracted from the webpage
    # Give me the value of source.The second argument is a fallback value. If source doesn't exist, give me "N/A" instead.
    print(f"Content length: {len (documents[0].page_content)} characters")
    print(f"Preview: {documents[0].page_content[:200]}...")

if __name__=="__main__":
    web_loader()

Loaded 1 document(s) from web
Source: https://en.wikipedia.org/wiki/Web_scraping
Content length: 31252 characters
Preview: 



Web scraping - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout Wiki...


In [7]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
import tempfile
import os


def lazy_loader():

    with tempfile.TemporaryDirectory() as tmpdir:

        # Create 5 text files
        for i in range(5):
            path = Path(tmpdir) / f"doc_{i}.txt"
            path.write_text(
                f"This is document {i}. It contains sample content."
            )

        # Create DirectoryLoader
        loader = DirectoryLoader(
            tmpdir,
            glob="*.txt",
            loader_cls=TextLoader
        )

        print("Initializing lazy loader for directory:", tmpdir)

        # Load documents one by one
        for doc in loader.lazy_load():
            print("Document Content Preview:", doc.page_content[:50], "...")
            print("Metadata:", doc.metadata["source"])


lazy_loader()

Initializing lazy loader for directory: C:\Users\DELL\AppData\Local\Temp\tmpv2syo3k3
Document Content Preview: This is document 0. It contains sample content. ...
Metadata: C:\Users\DELL\AppData\Local\Temp\tmpv2syo3k3\doc_0.txt
Document Content Preview: This is document 1. It contains sample content. ...
Metadata: C:\Users\DELL\AppData\Local\Temp\tmpv2syo3k3\doc_1.txt
Document Content Preview: This is document 2. It contains sample content. ...
Metadata: C:\Users\DELL\AppData\Local\Temp\tmpv2syo3k3\doc_2.txt
Document Content Preview: This is document 3. It contains sample content. ...
Metadata: C:\Users\DELL\AppData\Local\Temp\tmpv2syo3k3\doc_3.txt
Document Content Preview: This is document 4. It contains sample content. ...
Metadata: C:\Users\DELL\AppData\Local\Temp\tmpv2syo3k3\doc_4.txt


#UNDERSTANDING A DOCUMNET CLASS STRUCTURE

Documents are immutable, but we can always create new ones

In [ ]:
#Document() is a container/object used to hold a piece of text along with information about where that text came from.
from langchain_core.documents import Document

doc = Document(
    page_content="This is a manually created document.",
    metadata={
        "source": "manual_creation.txt",
        "author": "Paulo",
        "length": 30, ## A value YOU define
        "length_unit": "characters", #defining what length means explicitly.
        "tags": ["sample", "test"], #tags is a list of labels/keywords that you choose to associate with the document.The purpose is usually to help categorize, filter, or identify documents.
        "created_at": "2024-06-01",
    }
)

print("Document Structure:")
print(f" page_content (type): {type(doc.page_content)}")
print(f" page_content: {doc.page_content}")
print(f" metadata: {doc.metadata}")

#Document is immutable, Create a new Document and store it in updated_doc
updated_doc = Document (
    page_content=doc.page_content + "Additional content.", #string concatenation.
    metadata={**doc.metadata, "updated": True}, #** here Unpack all the key-value pairs from this dictionary.
)
#The ** here is called dictionary unpacking.If you do not use this and just do "updated": True the you would  lose the original metadata.

print(f"\n Updated metadata: {updated_doc.metadata}")
print(f" page_content (type): {type(doc.page_content)}")
print(f" page_content: {doc.page_content}")
print(f" metadata: {doc.metadata}")
I

Document Structure:
 page_content (type): <class 'str'>
 page_content: This is a manually created document.
 metadata: {'source': 'manual_creation.txt', 'author': 'Paulo', 'length': 30, 'tags': ['sample', 'test'], 'created_at': '2024-06-01'}


enumerate() is a built-in Python function that lets you loop over a sequence while getting both the index and the value at the same time.